In [8]:
import pandas as pd

train = pd.read_csv("../processed/train.csv")
val   = pd.read_csv("../processed/val.csv")
test  = pd.read_csv("../processed/test.csv")

print(train.shape)
print(train.head())

(350, 20)
   segment_length_km  is_monsoon_zone  is_midc_zone  is_bridge  hour_of_day  \
0           0.043206                0             0          1            4   
1           0.042877                0             0          0            3   
2           0.039029                0             0          0           22   
3           0.124607                0             0          0           12   
4           0.095345                0             0          0           23   

   day_of_week  is_peak_hour  base_speed_kmh  congestion_multiplier  \
0            0             0              40               0.894100   
1            0             0              40               0.845136   
2            6             0              40               0.988584   
3            2             0              40               0.742754   
4            3             0              40               0.863770   

   eta_minutes  road_type_living_street  road_type_primary  \
0     0.872574            

In [9]:
TARGET = "eta_minutes"

# Everything except eta_minutes = input features
feature_cols = [c for c in train.columns if c != TARGET]

X_train = train[feature_cols]
y_train = train[TARGET]

X_val = val[feature_cols]
y_val = val[TARGET]

X_test = test[feature_cols]
y_test = test[TARGET]

print("Features:", X_train.shape)
print("Target sample:", y_train.head())

Features: (350, 19)
Target sample: 0    0.872574
1    0.500000
2    0.500000
3    0.500000
4    0.500000
Name: eta_minutes, dtype: float64


In [10]:
X_train = X_train.replace({"True": 1, "False": 0})
X_val   = X_val.replace({"True": 1, "False": 0})
X_test  = X_test.replace({"True": 1, "False": 0})

In [11]:
# Convert categorical → numeric
X_train = pd.get_dummies(X_train)
X_val   = pd.get_dummies(X_val)
X_test  = pd.get_dummies(X_test)

# Align columns
X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Train model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Validate
val_preds = model.predict(X_val)
mae = mean_absolute_error(y_val, val_preds)
print(f"Validation MAE: {mae:.3f} minutes")

Validation MAE: 0.414 minutes


In [12]:
test_preds = model.predict(X_test)

if y_test is not None:
    from sklearn.metrics import mean_absolute_error
    test_mae = mean_absolute_error(y_test, test_preds)
    print(f"Test MAE: {test_mae:.3f} minutes")
else:
    print("Test predictions generated (no ground truth available).")

Test MAE: 0.393 minutes


In [13]:
print([col for col in X_train.columns if "eta" in col.lower()])

[]


In [14]:
print(pd.DataFrame({
    "Actual": y_test[:10],
    "Predicted": test_preds[:10]
}))

     Actual  Predicted
0  0.500000   1.097649
1  0.500000   1.000477
2  0.500000   0.834193
3  0.992482   0.693738
4  1.273198   1.358376
5  0.534306   1.569026
6  0.500000   0.935131
7  0.500000   0.751647
8  1.009599   1.124647
9  0.961360   0.619206


In [15]:
print(X_train.shape, X_test.shape)

(350, 19) (75, 19)


In [16]:
import joblib
import os

os.makedirs("models/trained", exist_ok=True)
joblib.dump(model, "models/trained/rf_model.pkl")
print("Model saved successfully!") 

Model saved successfully!


In [17]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [19]:
# LSTM needs scaled numbers between 0 and 1
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# LSTM expects 3D shape: (samples, timesteps, features)
# We treat each row as 1 timestep
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_val_lstm   = X_val_scaled.reshape(X_val_scaled.shape[0], 1, X_val_scaled.shape[1])
X_test_lstm  = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

print("Train shape:", X_train_lstm.shape)
print("Val shape:  ", X_val_lstm.shape)
print("Test shape: ", X_test_lstm.shape)

Train shape: (350, 1, 19)
Val shape:   (75, 1, 19)
Test shape:  (75, 1, 19)


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

model_lstm = Sequential([
    Input(shape=(1, X_train_scaled.shape[1])),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mae')
model_lstm.summary()

history = model_lstm.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        21,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,617 (92.25 KB)

 Trainable params: 23,617 (92.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - loss: 0.7518 - val_loss: 0.6281
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5685 - val_loss: 0.4180
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.3903 - val_loss: 0.3390
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.3870 - val_loss: 0.3470
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.3741 - val_loss: 0.3247
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.3638 - val_loss: 0.3175
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.3576 - val_loss: 0.3178
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3559 - val_loss: 0.3158
Epoch 9/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.3513 - val_loss: 0.3153
Epoch 10/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.3492 - val_loss: 0.3127


In [22]:
# Validation MAE
val_preds_lstm = model_lstm.predict(X_val_lstm).flatten()
val_mae_lstm = mean_absolute_error(y_val, val_preds_lstm)
print(f"LSTM Validation MAE: {val_mae_lstm:.3f} minutes")

# Test MAE
test_preds_lstm = model_lstm.predict(X_test_lstm).flatten()
test_mae_lstm = mean_absolute_error(y_test, test_preds_lstm)
print(f"LSTM Test MAE:       {test_mae_lstm:.3f} minutes")

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step
LSTM Validation MAE: 0.313 minutes
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
LSTM Test MAE:       0.346 minutes


In [23]:
import os
os.makedirs("models/trained", exist_ok=True)
model_lstm.save("models/trained/lstm_baseline.h5")
print("LSTM model saved at models/trained/lstm_baseline.h5 ✅")

LSTM model saved at models/trained/lstm_baseline.h5 ✅
